<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [2]:
!pip install -q tensorflow-recommenders tf-keras

In [3]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.20.0
TFRS: v0.7.7


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [3]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"
FEATURE_PATH = "/content/drive/MyDrive/Recommendation_Engine/features"
VOCAB_PATH = "/content/drive/MyDrive/Recommendation_Engine/vocabularies"

In [4]:
customers_df = spark.read.parquet(f"{PROCESSED_PATH}/customers_clean.parquet")

articles_df = spark.read.parquet(f"{PROCESSED_PATH}/articles_clean.parquet")

transactions_df = spark.read.parquet(f"{PROCESSED_PATH}/transactions_clean.parquet")

In [ ]:
recency_df = spark.read.parquet(f"{FEATURE_PATH}/recency.parquet")

product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/product_popularity.parquet"
)

monthly_product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/monthly_product_popularity.parquet"
)

In [6]:
customer_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

product_type_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/product_type_vocab.parquet"
)

department_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/department_vocab.parquet"
)

color_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/color_vocab.parquet"
)

In [7]:
print("Processed Datasets")
print("------------------")
print("Customers      :", customers_df.count())
print("Articles       :", articles_df.count())
print("Transactions   :", transactions_df.count())

print("\nFeature Tables")
print("------------------")
print("Recency                :", recency_df.count())
print("Product Popularity     :", product_popularity_df.count())
print("Monthly Popularity     :", monthly_product_popularity_df.count())

print("\nVocabularies")
print("------------------")
print("Customer Vocabulary    :", customer_vocab.count())
print("Article Vocabulary     :", article_vocab.count())
print("Product Type Vocabulary:", product_type_vocab.count())
print("Department Vocabulary  :", department_vocab.count())
print("Color Vocabulary       :", color_vocab.count())

Processed Datasets
------------------
Customers      : 1371980
Articles       : 105542
Transactions   : 31788324

Feature Tables
------------------
Recency                : 1362281
Product Popularity     : 104547
Monthly Popularity     : 768883

Vocabularies
------------------
Customer Vocabulary    : 1371980
Article Vocabulary     : 105542
Product Type Vocabulary: 131
Department Vocabulary  : 250
Color Vocabulary       : 50


#Create the Interaction Dataset

In [8]:
interactions_df = transactions_df.select(
    "customer_id",
    "article_id"
)

print("Total Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Total Interactions: 31788324
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016003 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016001 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|682236013 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016016 |
|aaa7a0483dd5b9e395d95324dcbfeb617af9800f39487d4b6aaee662bcd384c7|783335003 |
|aaa7b371465a823fec4312ef0f2807f924d54e5d41afb686223b76266bd9c599|563519008 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|783056001 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|695325016 |
|aaa8f491632b9022bf20aa444c793bdf23621bd9463c050e86b73ada4cb059b6|757333001 |
|aaa8f491632b9022bf20aa444c793bdf23

#Remove Duplicate User–Item Pairs

In [9]:
interactions_df = interactions_df.dropDuplicates()

print("Unique User-Item Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Unique User-Item Interactions: 27306439
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaac535f79b71437632d6001ebd960766da3000d7c455cd7a0500dab24cfd50e|799507001 |
|abd2537848661862039af47c8bcee2f9620c5fdc3b13c1af1f66f1f485c51de7|399087021 |
|ac2a5d7aa83653f77dbb343a90ebb705fb3c0a1e2683dbf74b1e08dab042c9bd|821152001 |
|ac43e628b476ec53ee48233d5ff7ad26d91b114095b51c8bdf8f5b560d101218|835730001 |
|acd03ec982613dcc026b69a4323f1db291cfbcd6a20173fab14c1063b7c014f2|708473003 |
|acfcd9df9a2a130cc54f547ea5f5829c5ff1913c9fda8e5b29a7badcbf544e26|737222004 |
|ad410adca76cb24d968194c0c2cf02d4714c2b8a639c9377c61020dc1972e8ef|734623002 |
|adb4d1ca1ae86f0a4592ba7ee5d586662945a45bb8d5a76761d971538f2c7980|728703008 |
|adceb8b35d5250062e3bd8b2a5025ee782874c798227c143ef6691488c75fb4d|399223001 |
|adf5b91a4a8092d8f2ce64e

In [10]:
from pyspark.sql.functions import rand

training_sample = (
    interactions_df
    .orderBy(rand())
    .sample(withReplacement=False, fraction=0.05, seed=42)
)

print("Training Sample Size:", training_sample.count())

training_sample.show(10, truncate=False)

Training Sample Size: 1366056
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|40fdc7f35a3ff5d7fa39250ff633a72476a095c3336460991252450de729cff9|600886007 |
|a1e1643d74793fa8ba0fd01dc32b8771c205f77198d1bffa81e55cb886fdc053|853545001 |
|3494eb620dddf8f3cc73b9dce39a77af77bfea0a4aa2710d05c8892dd401f4d4|716947001 |
|19ba775f3f25a397bfd279043459818b0c6f0bf435a7a7d173edb78541de0074|624486001 |
|c9d740a94330b334411248fed4f4d47b84a3ac09de2e07abfda61d69ef59079d|399256005 |
|157c4fea6694cffb710c6706dbbbd2880a394e136a606c065d8e3732e1f23ca4|800016002 |
|c5b0f2f719242f1dc0c438d13b5db164df374b98362c8592569ab50efc147717|474461047 |
|cd2b3b952ee7983c1cb0135823fdb04141c5fbd18095a019ba1083bdbb89376f|610776022 |
|7328ec6812fe89f071e50e3cb5ae1d2aa677a722ed9a0d533da80b061ca8cc47|654046004 |
|9fc20b51333990d63f7a4337914c591a2

In [11]:
training_pd = training_sample.toPandas()

customer_ids = training_pd["customer_id"].astype(str).values
article_ids = training_pd["article_id"].astype(str).values

print(customer_ids[:5])
print(article_ids[:5])

['40fdc7f35a3ff5d7fa39250ff633a72476a095c3336460991252450de729cff9'
 'a1e1643d74793fa8ba0fd01dc32b8771c205f77198d1bffa81e55cb886fdc053'
 '3494eb620dddf8f3cc73b9dce39a77af77bfea0a4aa2710d05c8892dd401f4d4'
 '19ba775f3f25a397bfd279043459818b0c6f0bf435a7a7d173edb78541de0074'
 'c9d740a94330b334411248fed4f4d47b84a3ac09de2e07abfda61d69ef59079d']
['600886007' '853545001' '716947001' '624486001' '399256005']


In [12]:
import tensorflow as tf

interactions_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": customer_ids,
    "article_id": article_ids
})

In [13]:
for sample in interactions_ds.take(5):
    print(sample)

{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'40fdc7f35a3ff5d7fa39250ff633a72476a095c3336460991252450de729cff9'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'600886007'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'a1e1643d74793fa8ba0fd01dc32b8771c205f77198d1bffa81e55cb886fdc053'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'853545001'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'3494eb620dddf8f3cc73b9dce39a77af77bfea0a4aa2710d05c8892dd401f4d4'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'716947001'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'19ba775f3f25a397bfd279043459818b0c6f0bf435a7a7d173edb78541de0074'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'624486001'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'c9d740a94330b334411248fed4f4d47b84a3ac09de2e07abfda61d69ef59079d'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'399256005'>}


In [14]:
BATCH_SIZE = 8192

train_ds = (
    interactions_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [15]:
customer_ids_vocab = (
    customer_vocab
    .select("customer_id")
    .toPandas()["customer_id"]
    .astype(str)
    .tolist()
)

article_ids_vocab = (
    article_vocab
    .select("article_id")
    .toPandas()["article_id"]
    .astype(str)
    .tolist()
)

print("Customers:", len(customer_ids_vocab))
print("Articles :", len(article_ids_vocab))

Customers: 1371980
Articles : 105542


In [16]:
customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_ids_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_ids_vocab,
    mask_token=None
)

In [17]:
sample_customer = customer_ids[0]
sample_article = article_ids[0]

print("Customer Index:", customer_lookup(tf.constant(sample_customer)).numpy())
print("Article Index :", article_lookup(tf.constant(sample_article)).numpy())

Customer Index: 1036502
Article Index : 22927


In [18]:
query_tower = tf.keras.Sequential([
    customer_lookup,

    tf.keras.layers.Embedding(
        input_dim=customer_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

In [19]:
sample_embedding = query_tower(
    tf.constant([customer_ids[0]])
)

print(sample_embedding.shape)
print(sample_embedding.numpy())

(1, 64)
[[-0.01151168  0.00810651 -0.01390479  0.00224275  0.00317861  0.00014357
  -0.02138322 -0.00862566  0.02346359 -0.00327776 -0.00720194  0.01492163
  -0.01220311  0.00748615 -0.00464399  0.00231813  0.01449986 -0.00796852
   0.00966839 -0.01608087 -0.00589836  0.0254504   0.02158787  0.01098604
   0.02044487  0.00437669  0.00705242 -0.00399092 -0.02745594  0.00874394
   0.01428234  0.00763705 -0.0139857  -0.00029645 -0.00797844  0.00691791
   0.01509579  0.0042242  -0.01724893  0.0246606  -0.02352318  0.00012076
  -0.01961647  0.01077048 -0.00725613  0.00127859 -0.00093762  0.00011983
  -0.00288732  0.0055381  -0.00232546  0.02072052  0.00779481 -0.02031577
  -0.02090469  0.01783461 -0.0110863  -0.02377021 -0.00172834 -0.01521966
  -0.01429485 -0.02778082 -0.03620724 -0.01042022]]


In [20]:
candidate_tower = tf.keras.Sequential([
    article_lookup,

    tf.keras.layers.Embedding(
        input_dim=article_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

In [21]:
sample_item_embedding = candidate_tower(
    tf.constant([article_ids[0]])
)

print(sample_item_embedding.shape)
print(sample_item_embedding.numpy())

(1, 64)
[[ 0.00106113 -0.00607625 -0.00580419 -0.01944447 -0.03789406 -0.00465081
   0.0110612  -0.02763123 -0.01245098 -0.01150334  0.01700091  0.03053629
  -0.00123545 -0.01727846 -0.00748236  0.01373649 -0.00411552  0.00880716
  -0.02937938  0.02912002 -0.00355918  0.00332838  0.01768938  0.01805261
  -0.01248198  0.00334385  0.01023934 -0.01462938  0.00639922 -0.00389827
   0.01245279  0.01122691 -0.01180455 -0.02188231 -0.00037541  0.00761145
  -0.02046042  0.01696289 -0.00255221 -0.00659826  0.0145181   0.00248224
   0.00733311  0.02782681 -0.00610492 -0.00969078  0.00873767 -0.01035673
  -0.00301584  0.01582415  0.01809919  0.00485391  0.00569219 -0.00826475
  -0.0309652  -0.01891085 -0.02552386  0.01605136  0.00883624 -0.00336408
  -0.00493105 -0.0020676  -0.00417484  0.02207923]]


In [22]:
print("User Embedding Shape :", sample_embedding.shape)
print("Item Embedding Shape :", sample_item_embedding.shape)

User Embedding Shape : (1, 64)
Item Embedding Shape : (1, 64)


In [23]:
candidate_dataset = tf.data.Dataset.from_tensor_slices(
    article_ids_vocab
).batch(1024)

In [24]:
for item in candidate_dataset.take(1):
    print(item[:5])

tf.Tensor([b'108775015' b'108775044' b'108775051' b'110065001' b'110065002'], shape=(5,), dtype=string)
